# Week 5: Case - based GAN

Train a Generative Adversarial Network using the [anime faces dataset by MckInsey666](https://github.com/bchao1/Anime-Face-Dataset).

Name: Muhammad Rafly Arjasubrata (203012420028) & Muhammad Abiya Makruf (203012420034)

## Imports

In [ ]:
import torch
import time
import pandas as pd

from anime_gan.config import LSGANConfig
from anime_gan.data import create_dataloader, download_dataset
from anime_gan.metrics import evaluate_gan_models, plot_metrics_comparison, print_metrics_table
from anime_gan.training import train_gan
from anime_gan.visualization import (
    plot_final_samples,
    plot_gan_comparison,
    plot_side_by_side_fixed_samples,
    plot_training_dashboard,
)

## Parameters

In [ ]:
cfg = LSGANConfig()
cfg.set_seed()
device = cfg.get_device()
cfg.create_dirs()

print(f"Using device: {device}")
print(f"Data root: {cfg.data_root}")
print(f"Images dir: {cfg.images_dir}")
print(f"Checkpoints dir: {cfg.checkpoints_dir}")

## Dataset Download and Preperation

Download the Anime Faces dataset and save it to a local directory. Source dataset: https://storage.googleapis.com/learning-datasets/Resources/anime-faces.zip

In [ ]:
download_dataset(
    image_dir=cfg.image_dir,
    zip_name=cfg.zip_name,
    data_root=cfg.data_root,
    data_url=cfg.data_url,
)

print(f"Image directory: {cfg.image_dir}")
print(f"Total images found: {len(list(cfg.image_dir.glob('*')))}")

In [ ]:
dataset, dataloader = create_dataloader(
    image_dir=cfg.image_dir,
    device=device,
    image_size=cfg.img_size,
    batch_size=cfg.batch_size,
    n_cpu=cfg.n_cpu,
)

print(f"Dataset size: {len(dataset)}")
print(f"Batches per epoch: {len(dataloader)}")

## Training Vanilla GAN & LSGAN

We will train 2 models in this notebook:
- Vanilla GAN (baseline)
- LSGAN (least squares GAN)

This can take long time because training runs two times.
Both models use same dataset and same main hyperparameters for fair compare.

### Vanilla GAN

In [ ]:
cfg.set_seed()

start_time_vanilla = time.time()

vanilla_result = train_gan(
    cfg=cfg,
    dataloader=dataloader,
    device=device,
    gan_type="vanilla",
    run_name="vanilla",
)

end_time_vanilla = time.time()

### LSGAN

In [ ]:
cfg.set_seed()

start_time_LSGAN = time.time()

lsgan_result = train_gan(
    cfg=cfg,
    dataloader=dataloader,
    device=device,
    gan_type="lsgan",
    run_name="lsgan",
)

end_time_LSGAN = time.time()

### Training history

In [ ]:
vanilla_history = vanilla_result["history"]
lsgan_history = lsgan_result["history"]

vanilla_generator = vanilla_result["generator"]
lsgan_generator = lsgan_result["generator"]

comparison_fixed_noise = torch.randn(cfg.fixed_sample_size, cfg.latent_dim, device=device)

print("Training finished for both models")
print(f"Vanilla best epoch: {vanilla_result['best_epoch']} | best margin: {vanilla_result['best_margin']:.4f}")
print(f"LSGAN best epoch: {lsgan_result['best_epoch']} | best margin: {lsgan_result['best_margin']:.4f}")

Vanilla training time: 0.00 seconds
LSGAN training time: 0.00 seconds


In [ ]:
time_comparison_df = pd.DataFrame({
    "Model": ["Vanilla GAN", "LSGAN"],
    "Training Time (seconds)": [
        end_time_vanilla - start_time_vanilla,
        end_time_LSGAN - start_time_LSGAN,
    ]
})

display(time_comparison_df)

,Model,Training Time (seconds)
0,Vanilla GAN,0.0
1,LSGAN,0.0


## Metric Comparison

- FID (lower is better)
- KID (lower is better)
- LPIPS diversity (higher means more variation)

These numbers help compare Vanilla and LSGAN in fair way.

In [ ]:
plot_training_dashboard(
    g_losses=vanilla_history["g_losses"],
    d_losses=vanilla_history["d_losses"],
    d_real_scores=vanilla_history["d_real_scores"],
    d_fake_scores=vanilla_history["d_fake_scores"],
    epoch_margins=vanilla_history["epoch_margins"],
    epoch_lr_g=vanilla_history["epoch_lr_g"],
    epoch_lr_d=vanilla_history["epoch_lr_d"],
    best_margin=vanilla_result["best_margin"],
    save_prefix="vanilla",
    title_prefix="Vanilla GAN",
    images_dir=str(cfg.images_dir),
)

plot_training_dashboard(
    g_losses=lsgan_history["g_losses"],
    d_losses=lsgan_history["d_losses"],
    d_real_scores=lsgan_history["d_real_scores"],
    d_fake_scores=lsgan_history["d_fake_scores"],
    epoch_margins=lsgan_history["epoch_margins"],
    epoch_lr_g=lsgan_history["epoch_lr_g"],
    epoch_lr_d=lsgan_history["epoch_lr_d"],
    best_margin=lsgan_result["best_margin"],
    save_prefix="lsgan",
    title_prefix="LSGAN",
    images_dir=str(cfg.images_dir),
)

plot_gan_comparison(
    vanilla_history=vanilla_history,
    lsgan_history=lsgan_history,
    images_dir=str(cfg.images_dir),
)

plot_final_samples(
    generator=vanilla_generator,
    fixed_noise=comparison_fixed_noise,
    latent_dim=cfg.latent_dim,
    device=device,
    save_prefix="vanilla",
    title_prefix="Vanilla GAN",
    images_dir=str(cfg.images_dir),
)

plot_final_samples(
    generator=lsgan_generator,
    fixed_noise=comparison_fixed_noise,
    latent_dim=cfg.latent_dim,
    device=device,
    save_prefix="lsgan",
    title_prefix="LSGAN",
    images_dir=str(cfg.images_dir),
)

plot_side_by_side_fixed_samples(
    vanilla_generator=vanilla_generator,
    lsgan_generator=lsgan_generator,
    fixed_noise=comparison_fixed_noise,
    images_dir=str(cfg.images_dir),
)

metrics = evaluate_gan_models(
    vanilla_generator=vanilla_generator,
    lsgan_generator=lsgan_generator,
    dataloader=dataloader,
    device=device,
    latent_dim=cfg.latent_dim,
    max_samples=2048,
    lpips_samples=128,
    lpips_pairs=64,
)

print_metrics_table(metrics)
plot_metrics_comparison(metrics, images_dir=str(cfg.images_dir))

print("Compare done")
print(f"Vanilla best margin: {vanilla_result['best_margin']:.4f}")
print(f"LSGAN best margin: {lsgan_result['best_margin']:.4f}")